# Analyze Features Notebook
Use this notebook to interactively run the feature-analysis workflow without editing one long script.

Workflow:
1. Run the setup cell.
2. Edit the parameters cell to choose feature dataframes and plot groups.
3. Run the helper cells.
4. Run the analysis cell.
5. Use the inspection cell to preview generated outputs and selected feature tables.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

cwd = Path.cwd()
if (cwd / "analyze_features.ipynb").exists():
    NOTEBOOK_DIR = cwd
elif (cwd / "data_proc_2d" / "app" / "analyze_features.ipynb").exists():
    NOTEBOOK_DIR = cwd / "data_proc_2d" / "app"
else:
    NOTEBOOK_DIR = cwd

PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "app" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "data_proc_2d") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "data_proc_2d"))

from utilities import file_io, log_utils
from src.file_io_utils import iter_files, load_step_ids_from_json, load_torch
from src import pose_analysis_yolo, plot_utils
from data_proc_2d.src import feature_extraction
from src.yolo_pose_config import JOINT_ANGLE_TRIPLETS, JOINT_ANGLE_TRIPLETS_CAL, RATIO_BETWEEN_DISTS

PROJECT_ROOT

WindowsPath('c:/Users/Owner/OneDrive - Universität Stuttgart/2025_26_Thesis/codes/LSTM_HRC')

## Editable Parameters
Update this cell when you want to change which feature dataframes are plotted, limit the input files, switch between per-video and cross-video outputs, or choose whether plots are shown inline or saved to disk.

In [14]:
IS_SOURCE_PT = False
VIDEO_FILTER = ["cam-01_uid-01"]  # Example: ["cam-01_uid-01"] or None
MAX_FILES = None

FEATURE_PANEL_CONFIG = [
    ("velocity_scale_db", "Speed (norm. units / frame)", "Joint Speed", "log"),
    ("acceleration_scale_db", "Acceleration (norm. units / frame²)", "Joint Acceleration", "log"),
    ("velocity_x_db", "Velocity component X (norm. units / frame)", "Joint Velocity X", "log"),
    ("velocity_y_db", "Velocity component Y (norm. units / frame)", "Joint Velocity Y", "log"),
    ("acceleration_x_db", "Acceleration component X (norm. units / frame²)", "Joint Acceleration X", "log"),
    ("acceleration_y_db", "Acceleration component Y (norm. units / frame²)", "Joint Acceleration Y", "log"),
    ("pol_vectors_x_db", "Polar vector component X", "Polar Vectors X", "linear"),
    ("pol_vectors_y_db", "Polar vector component Y", "Polar Vectors Y", "linear"),
    ("pol_angles_db", "Polar angle (deg)", "Polar Angles", "linear"),
    ("joint_angles_db", "Angle (deg)", "Joint Angles", "linear"),
    ("ratios_db", "Ratio", "Distance Ratios", "log"),
    ("dist_ratios_db", "Distance from center ratio", "Distance from Center", "linear"),
]

SELECTED_FEATURE_DFS = [
    "velocity_scale_db",
    "acceleration_scale_db",
    "velocity_x_db",
    "velocity_y_db",
    "acceleration_x_db",
    "acceleration_y_db",
    "pol_vectors_x_db",
    "pol_vectors_y_db",
    "pol_angles_db",
    "joint_angles_db",
    "ratios_db",
    "dist_ratios_db",
]

RUN_SINGLE_VIDEO_PLOTS = True
RUN_ALL_VIDEOS_PER_FEATURE = True
SHOW_PLOTS_IN_NOTEBOOK = True
SAVE_PLOTS = False

IO_ROOT = PROJECT_ROOT / "data_proc_2d" / "results" / "video_analysis_yolo"
DB_PATH = Path(r"G:\.shortcut-targets-by-id\1Ykdzx6UjCe0KPKy_6M4LgCOTxKK6Awgy\Videos")
PT_ROOT = PROJECT_ROOT / "data_proc_2d" / "dataset" / "train" / "raw" if IS_SOURCE_PT else DB_PATH / "dataset" / "train" / "raw"
LABEL_ROOT = DB_PATH / "archive" / "annotations"
ANNO_ROOT = DB_PATH / "dataset" / "annotations"
PLOT_DIR = IO_ROOT / "plots"
CSV_DIR = IO_ROOT / "source"
OUTPUT_JSON = IO_ROOT / "source" / "feature_results.json"

for path in (CSV_DIR,):
    path.mkdir(parents=True, exist_ok=True)
if SAVE_PLOTS:
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

selected_panel_names = SELECTED_FEATURE_DFS or [cfg[0] for cfg in FEATURE_PANEL_CONFIG]
display(
    pd.DataFrame(
        {
            "selected_feature_df": selected_panel_names,
            "show_in_notebook": [SHOW_PLOTS_IN_NOTEBOOK] * len(selected_panel_names),
            "save_plot_files": [SAVE_PLOTS] * len(selected_panel_names),
        }
    )
)

,selected_feature_df,show_in_notebook,save_plot_files
0,velocity_scale_db,True,False
1,acceleration_scale_db,True,False
2,velocity_x_db,True,False
3,velocity_y_db,True,False
4,acceleration_x_db,True,False
5,acceleration_y_db,True,False
6,pol_vectors_x_db,True,False
7,pol_vectors_y_db,True,False
8,pol_angles_db,True,False
9,joint_angles_db,True,False


## Data And Feature Helpers
These helper functions load annotations, extract feature tensors from each PT file, and normalize video names used for per-video and cross-video plots.

In [11]:
def get_notebook_logger():
    logger_name = "pose_processing_notebook"
    logger = log_utils.setup_logger(logger_name)
    log_path = PROJECT_ROOT / "logs" / "video_analysis_notebook.log"
    absolute_log_path = str(log_path.resolve())
    if not any(getattr(handler, "baseFilename", None) == absolute_log_path for handler in logger.handlers):
        log_utils.save_log_to_file(logger, absolute_log_path)
    return logger


def get_selected_panel_config(selected_feature_dfs=None):
    if not selected_feature_dfs:
        return list(FEATURE_PANEL_CONFIG)
    selected = set(selected_feature_dfs)
    return [config for config in FEATURE_PANEL_CONFIG if config[0] in selected]


def load_annotation_maps(logger):
    step_label_list = {}
    for label_file in iter_files(LABEL_ROOT, extension=".json"):
        label_info, video_file_name = load_step_ids_from_json(label_file, logger=logger)
        step_label_list[video_file_name] = {"labels": label_info}

    elan_step_label_list = {}
    for label_file in iter_files(ANNO_ROOT, extension=".json"):
        video_file_name = f"video__{label_file.stem.split('__')[1]}.mp4"
        label_info = file_io.load_json(str(label_file), logger=logger)
        elan_step_label_list[video_file_name] = {"labels": label_info}

    return step_label_list, elan_step_label_list


def find_annotations(metadata, step_label_list, elan_step_label_list, logger):
    video_path = metadata.get("video")
    video_file_name = os.path.basename(video_path) if video_path else "unknown"

    annotations = {}
    step_labels = step_label_list.get(video_file_name, {}).get("labels", [])
    elan_step_labels = elan_step_label_list.get(video_file_name, {}).get("labels", [])

    if step_labels:
        logger.info("Found %d step labels for video %s", len(step_labels), video_file_name)
        annotations["step_labels"] = step_labels
    if elan_step_labels:
        logger.info("Found %d ELAN step labels for video %s", len(elan_step_labels), video_file_name)
        annotations["elan_step_labels"] = elan_step_labels

    return annotations


def extract_features(pt_file, logger):
    data = load_torch(str(pt_file), logger=logger)

    metadata = data.get("metadata", {})
    landmarks = data["smoothed_landmarks"]  # Shape: (num_frames, num_joints, 2)
    features = {"metadata": metadata}

    landmarks = feature_extraction._log_malicious_tensor(landmarks, "landmarks")
    landmarks = feature_extraction._fill_zero_frames_with_previous(landmarks, "landmarks")

    velocity_scale, acceleration_scale, velocity_xy, acceleration_xy = feature_extraction.veclocity_acceleration_magnitude(landmarks)
    features["velocity_scale"] = velocity_scale
    features["acceleration_scale"] = acceleration_scale
    features["velocity_x"] = velocity_xy[:, :, 0]
    features["velocity_y"] = velocity_xy[:, :, 1]
    features["acceleration_x"] = acceleration_xy[:, :, 0]
    features["acceleration_y"] = acceleration_xy[:, :, 1]

    vectors, angles = feature_extraction.polar_coordinate_features(landmarks)
    features["pol_vectors_x"] = vectors[:, :, 0]
    features["pol_vectors_y"] = vectors[:, :, 1]
    features["pol_angles"] = angles

    joint_angles = []
    for _, a, vertex, c in JOINT_ANGLE_TRIPLETS:
        joint_angles.append(feature_extraction.angle_at_joint(landmarks, a, vertex, c))
    for _, a, b, c in JOINT_ANGLE_TRIPLETS_CAL:
        middle_point = (landmarks[:, a, :] + landmarks[:, b, :]) / 2
        joint_angles.append(feature_extraction.angle_at_joint(landmarks, a, middle_point, c))
    features["joint_angles"] = torch.stack(joint_angles, dim=1)

    distance_ratios = []
    for _, first_dist_feature, second_dist_feature in RATIO_BETWEEN_DISTS:
        distance_ratios.append(feature_extraction.distance_ratio(landmarks, first_dist_feature, second_dist_feature))
    features["ratios"] = torch.stack(distance_ratios, dim=1)
    features["dist_ratios"] = feature_extraction.distance_from_center(
        landmarks,
        ("left_hip", "right_hip", "left_shoulder", "right_shoulder"),
        ("left_hip", "right_hip"),
    )

    return features


def normalise_video_name(metadata_video, fallback):
    video_name = Path(metadata_video or fallback).stem
    return video_name.split("__", maxsplit=1)[1] if "__" in video_name else video_name


def select_pt_files():
    pt_files = list(iter_files(PT_ROOT, extension=".pt"))
    if VIDEO_FILTER:
        pt_files = [pt_file for pt_file in pt_files if any(f in pt_file.name for f in VIDEO_FILTER)]
    if MAX_FILES is not None:
        pt_files = pt_files[:MAX_FILES]
    return pt_files

## Plotting And Execution Helpers
This section keeps the plotting logic separate from the extraction logic, so you can change selected feature dataframes or rerun only the plotting steps more easily.

In [ ]:
def plot_single_video_features(features, annotations, metadata, logger, panel_config):
    feature_dfs = pose_analysis_yolo.build_feature_dataframes(features)
    features_db = pd.concat(feature_dfs.values(), axis=1) if feature_dfs else pd.DataFrame()
    panel_data = pose_analysis_yolo.build_panel_data(feature_dfs, panel_config)
    file_name = Path(metadata.get("video", "unknown_video")).stem

    if RUN_SINGLE_VIDEO_PLOTS and panel_data:
        plot_path = str(PLOT_DIR / f"{file_name}_features.png") if SAVE_PLOTS else None
        plot_utils.plot_features(
            panel_data=panel_data,
            annotations=annotations,
            save_path=plot_path,
            suptitle=f"Video: {file_name}",
            show=SHOW_PLOTS_IN_NOTEBOOK,
        )

    features_db.to_csv(str(CSV_DIR / f"{file_name}_features.csv"), index=False)

    return {
        "feature_dfs": feature_dfs,
        "summary": {
            "feature_dataframes": {
                dataframe_name: feature_df.columns.tolist()
                for dataframe_name, feature_df in feature_dfs.items()
            },
            "panel_titles": [panel.title for panel in panel_data],
            "feature_columns": features_db.columns.tolist(),
            "num_frames": int(features_db.shape[0]),
            "num_features": int(features_db.shape[1]),
            "annotations": annotations,
            "plot_saved": SAVE_PLOTS and RUN_SINGLE_VIDEO_PLOTS and bool(panel_data),
        },
    }


def plot_all_videos_per_feature(feature_dataframes_by_video, annotations_by_video, panel_config, logger):
    summary = {}
    if not RUN_ALL_VIDEOS_PER_FEATURE:
        return summary

    for dataframe_name, ylabel, panel_title, yscale in panel_config:
        prefixed_frames = []
        included_videos = []
        video_panels = []

        for video_name, feature_dataframes in feature_dataframes_by_video.items():
            feature_df = feature_dataframes.get(dataframe_name)
            if feature_df is None or feature_df.empty:
                continue

            prefixed_frames.append(feature_df.add_prefix(f"{video_name}__"))
            included_videos.append(video_name)
            video_panels.append(
                plot_utils.VideoFeaturePanels(
                    video_name=video_name,
                    feature_panel=plot_utils.PanelData(
                        df=feature_df,
                        names=list(feature_df.columns),
                        cols=list(feature_df.columns),
                        ylabel=ylabel,
                        title=f"{panel_title}: {video_name}",
                        yscale=yscale,
                    ),
                    annotations=annotations_by_video.get(video_name, {}),
                )
            )

        if not video_panels:
            continue

        combined_df = pd.concat(prefixed_frames, axis=1)
        plot_path = PLOT_DIR / f"{dataframe_name}_all_videos.png" if SAVE_PLOTS else None
        csv_path = CSV_DIR / f"{dataframe_name}_all_videos.csv"

        plot_utils.plot_features_by_video(
            video_panels=video_panels,
            save_path=str(plot_path) if plot_path else None,
            suptitle=f"{panel_title} Across All Videos",
            show=SHOW_PLOTS_IN_NOTEBOOK,
        )
        combined_df.to_csv(csv_path, index=True, index_label="frame")

        summary[dataframe_name] = {
            "plot_path": str(plot_path) if plot_path else None,
            "csv_path": str(csv_path),
            "num_videos": len(included_videos),
            "num_traces": int(len(combined_df.columns)),
            "videos": included_videos,
            "columns": list(combined_df.columns),
            "plot_saved": SAVE_PLOTS,
            "plot_shown": SHOW_PLOTS_IN_NOTEBOOK,
        }

    return summary


def run_analysis(selected_feature_dfs=None):
    logger = get_notebook_logger()
    panel_config = get_selected_panel_config(selected_feature_dfs)
    step_label_list, elan_step_label_list = load_annotation_maps(logger)
    pt_files = select_pt_files()

    logger.info("Found %d PT file(s) under %s", len(pt_files), PT_ROOT)

    results_summary = {}
    feature_dataframes_by_video = {}
    annotations_by_video = {}

    for pt_file in pt_files:
        logger.info("Processing %s", pt_file.name)
        pt_data = extract_features(pt_file, logger) if IS_SOURCE_PT else load_torch(str(pt_file), logger=logger)

        metadata = pt_data.get("metadata", {})
        features = {key: value for key, value in pt_data.items() if key != "metadata"}
        annotations = find_annotations(metadata, step_label_list, elan_step_label_list, logger)
        video_name = normalise_video_name(metadata.get("video"), pt_file.stem)

        single_video_result = plot_single_video_features(features, annotations, metadata, logger, panel_config)
        feature_dataframes_by_video[video_name] = single_video_result["feature_dfs"]
        annotations_by_video[video_name] = annotations
        results_summary[pt_file.stem] = single_video_result["summary"]

    results_summary["_all_videos_per_feature"] = plot_all_videos_per_feature(
        feature_dataframes_by_video,
        annotations_by_video,
        panel_config,
        logger,
    )
    file_io.save_json(results_summary, str(OUTPUT_JSON), logger=logger)

    return {
        "logger": logger,
        "panel_config": panel_config,
        "pt_files": pt_files,
        "results_summary": results_summary,
        "feature_dataframes_by_video": feature_dataframes_by_video,
        "annotations_by_video": annotations_by_video,
    }

## Run Analysis
Rerun this cell after changing the selected feature dataframes, the file filter, or the plotting flags.

In [13]:
analysis_state = run_analysis(selected_feature_dfs=SELECTED_FEATURE_DFS)

display(
    pd.DataFrame({
        "processed_file": [pt_file.name for pt_file in analysis_state["pt_files"]],
    })
)

2026-04-20 21:01:07,097 | INFO | pose_processing_notebook | Loaded JSON from G:\.shortcut-targets-by-id\1Ykdzx6UjCe0KPKy_6M4LgCOTxKK6Awgy\Videos\archive\annotations\cam-01\label__cam-01_uid-01_take-01.json
2026-04-20 21:01:07,098 | INFO | pose_processing_notebook | Loaded step marker: step_id=0, timestamp=4.167
2026-04-20 21:01:07,099 | INFO | pose_processing_notebook | Loaded step marker: step_id=1, timestamp=21.7
2026-04-20 21:01:07,100 | INFO | pose_processing_notebook | Loaded step marker: step_id=2, timestamp=42.7
2026-04-20 21:01:07,100 | INFO | pose_processing_notebook | Loaded step marker: step_id=3, timestamp=72.367
2026-04-20 21:01:07,101 | INFO | pose_processing_notebook | Loaded step marker: step_id=1, timestamp=86.267
2026-04-20 21:01:07,106 | INFO | pose_processing_notebook | Loaded JSON from G:\.shortcut-targets-by-id\1Ykdzx6UjCe0KPKy_6M4LgCOTxKK6Awgy\Videos\archive\annotations\cam-01\label__cam-01_uid-01_take-02.json
2026-04-20 21:01:07,107 | INFO | pose_processing_note

2026-04-20 21:01:07,115 | INFO | pose_processing_notebook | Loaded JSON from G:\.shortcut-targets-by-id\1Ykdzx6UjCe0KPKy_6M4LgCOTxKK6Awgy\Videos\archive\annotations\cam-01\label__cam-01_uid-01_take-03.json
2026-04-20 21:01:07,115 | INFO | pose_processing_notebook | Loaded step marker: step_id=0, timestamp=3.667
2026-04-20 21:01:07,116 | INFO | pose_processing_notebook | Loaded step marker: step_id=1, timestamp=16.6
2026-04-20 21:01:07,117 | INFO | pose_processing_notebook | Loaded step marker: step_id=2, timestamp=23.367
2026-04-20 21:01:07,117 | INFO | pose_processing_notebook | Loaded step marker: step_id=3, timestamp=39.767
2026-04-20 21:01:07,118 | INFO | pose_processing_notebook | Loaded step marker: step_id=1, timestamp=50.867
2026-04-20 21:01:07,123 | INFO | pose_processing_notebook | Loaded JSON from G:\.shortcut-targets-by-id\1Ykdzx6UjCe0KPKy_6M4LgCOTxKK6Awgy\Videos\archive\annotations\cam-01\label__cam-01_uid-01_take-04.json
2026-04-20 21:01:07,124 | INFO | pose_processing_no

KeyError: 'smoothed_landmarks'

## Inspect Outputs
Use this cell to preview the selected feature groups, inspect one sample feature table, and review the cross-video plot summary.

In [9]:
selected_feature_dfs = [cfg[0] for cfg in analysis_state["panel_config"]]
display(pd.DataFrame({"selected_feature_df": selected_feature_dfs}))

if analysis_state["feature_dataframes_by_video"] and selected_feature_dfs:
    sample_video = next(iter(analysis_state["feature_dataframes_by_video"]))
    sample_feature_df = selected_feature_dfs[0]
    display(
        analysis_state["feature_dataframes_by_video"][sample_video][sample_feature_df].head()
    )

aggregate_summary = analysis_state["results_summary"].get("_all_videos_per_feature", {})
if aggregate_summary:
    display(
        pd.DataFrame.from_dict(aggregate_summary, orient="index")[
            ["num_videos", "num_traces"]
        ]
    )

NameError: name 'analysis_state' is not defined